<a href="https://colab.research.google.com/github/Onar-Karadogan/rag-notebook/blob/main/SimpleRAGForFileQueries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai llama-index-vector-stores-chroma llama-parse chromadb

In [ ]:
import os
import chromadb
from pathlib import Path
from google.colab import userdata

# LlamaIndex Core & Google Integrations
from llama_index.core import Settings, VectorStoreIndex, StorageContext, SimpleDirectoryReader
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_parse import LlamaParse

class IngestionEngine:
    def __init__(self, db_path="./chroma_db"):
        """Initializes API keys from Colab Secrets and Global Settings."""
        # Fetch keys from Colab Secret Manager
        try:
            self.gemini_key = userdata.get("GOOGLE_API_KEY")
            self.llama_key = userdata.get("LLAMA_CLOUD_API_KEY")
        except Exception:
            raise ValueError("Missing API keys! Add GOOGLE_API_KEY and LLAMA_CLOUD_API_KEY to Colab Secrets.")

        # --- Global LLM & Embedding Config ---
        Settings.llm = GoogleGenAI(
            model="models/gemini-2.0-flash",
            api_key=self.gemini_key
        )

        Settings.embed_model = GoogleGenAIEmbedding(
            model_name="models/text-embedding-004",
            api_key=self.gemini_key
        )

        # --- LlamaParse Config ---
        self.parser = LlamaParse(
            api_key=self.llama_key,
            result_type="markdown"
        )

        self.db_path = db_path

    def process_and_index(self, file_path):
        """Parses file based on extension and creates searchable index."""
        print(f"--- Step 1: Parsing {Path(file_path).suffix} data ---")

        # Automatically map the parser to the file's extension
        ext = Path(file_path).suffix
        file_extractor = {ext: self.parser} if ext in [".pdf", ".docx", ".pptx", ".xlsx", ".csv"] else {}

        reader = SimpleDirectoryReader(
            input_files=[file_path],
            file_extractor=file_extractor
        )
        documents = reader.load_data()

        print(f"--- Step 2: Initializing Vector Store ---")
        db = chromadb.PersistentClient(path=self.db_path)
        chroma_collection = db.get_or_create_collection("document_knowledge")

        vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        index = VectorStoreIndex.from_documents(
            documents,
            storage_context=storage_context,
            show_progress=True
        )

        print(f"--- Ingestion Complete! ---")
        return index

# --- Execution Logic ---
engine = IngestionEngine()
file_to_process = "bilgisayar_sinav_prog_2025_2026guz.xlsx"

if os.path.exists(file_to_process):
    knowledge_index = engine.process_and_index(file_to_process)
    query_engine = knowledge_index.as_query_engine()

    # Notebook Example Query
    user_query = "Summarize the exam schedule" # Change this to test
    response = query_engine.query(user_query)
    print(f"\nGemini Response:\n{response}")
else:
    print(f"Error: File '{file_to_process}' not found. Please upload it to the Colab files pane.")